# LLM as a Judge Analysis

This notebook implements an LLM-as-a-judge approach to assess the reasoning quality of model outputs. It reads existing results from `strategy_*_results` directories, evaluates the `reasoning` column using a specific rubric, and saves the detailed annotations.

In [19]:
import os
import glob
import json
import asyncio
import pandas as pd
import numpy as np
from openai import AsyncAzureOpenAI

# Apply nest_asyncio to allow nested event loops in Jupyter
import nest_asyncio
nest_asyncio.apply()

In [20]:
# ---- Configuration ----

# API Configuration provided by user
API_KEY = 'MmI5OWMyOWRhMmFhNGFkNTljNDhjNTEzMTIwZWRjNzM6NTlhYjI3ZDdiQzM1NEFiMkI5MjNmRTcwOUM0QUYwZEQ='
# Reverted to validated version to fix 404 error
API_VERSION = '2024-10-21'
RESOURCE_ENDPOINT = 'https://unified-api.ucsf.edu/general'

# Model Settings
JUDGE_MODEL = "gpt-4.1-mini-2025-04-14"
JUDGE_TEMPERATURE = 0.0
JUDGE_SEED = 42

# Processing Settings
CONCURRENCY = 40  # Adjusted for reasonable parallel processing
MAX_RETRIES = 5
RATE_DELAY = 0.1

# Configuration to control scope
# Set TEST_SUBSET to an integer (e.g., 10) to test on a small random sample per file.
# Set to None to run on the full dataset.
TEST_SUBSET = None

# Input field to specify specific files to read in.
# If left empty, it will read all available files as is the current logic.
# Example: TARGET_FILES = ['strategy_one_results/results_openai_gpt-5-nano.csv']
TARGET_FILES = []

RESULTS_DIR = "llm_judge_results"

In [21]:
# ---- Client Setup ----
client = AsyncAzureOpenAI(
    api_key=API_KEY,
    api_version=API_VERSION,
    azure_endpoint=RESOURCE_ENDPOINT,
)

sem = asyncio.Semaphore(CONCURRENCY)

In [22]:
PROMPT_TEMPLATE = """
You are an expert annotation model evaluating reasoning texts from a pandemic policy study.

Context: In this study, participants chose between varying policy scenarios that differed in infection rates, business closure levels, hospital capacity constraints, and projected economic impact. Your task is to analyze their written justifications.

Goal: Extract and label explicit decision factors from the text using strict, literal definitions. Do not evaluate the quality or correctness of the decision. Instead, annotate only whether the text explicitly mentions the study's focal concepts—specifically infection outcomes, hospital capacity, economic impacts, business closures, and user preference—exactly as defined below.

### INPUT TEXT
{reasoning_text}

### DEFINITIONS & CRITERIA
Evaluate the text against the following 13 definitions. Be literal: if a concept is implied but not explicitly stated, mark it as False.

1. mentions_hospital_capacity
- TRUE if the text explicitly mentions hospital capacity, beds, ICU limits, or occupancy.
- FALSE if it only mentions "health system" generally without reference to physical capacity or beds.

2. computed_hospital_demand
- TRUE if the text performs an explicit numeric calculation (e.g., "10% of 200 infections = 20 beds needed").
- FALSE if it only mentions qualitative risk (e.g., "hospitalizations will rise") without showing the math/numbers.

3. compares_demand_to_capacity
- TRUE if the text explicitly compares demand vs. capacity using comparison language (e.g., "exceeds capacity," "overwhelm," "surpass," "remain within limits").
- FALSE if it mentions both figures but does not explicitly state their relationship (e.g., "Capacity is X and demand is Y" without saying "X > Y").

4. short_term_economic_reasoning
- TRUE if the text explicitly references immediate financial consequences, such as "income loss," "lost wages," "revenue loss," or "daily costs" to individuals or businesses.
- FALSE if the text mentions "economic impact" or "costs" generally without specifying *immediate* financial loss (e.g., "bad for the economy" is too vague).

5. long_term_economic_reasoning
- TRUE if the text explicitly references broader or lasting economic damage, such as "recession," "long-term recovery," "bankruptcy," "permanent closure," or "GDP impact."
- FALSE if the text mentions generic "economic consequences" or only immediate losses without linking them to *lasting* structural damage.

6. mentions_business_closures
- TRUE if the text explicitly references the policy of closing businesses, shutdowns, lockdowns, or keeping venues open.
- FALSE if the text only mentions "reducing activity," "social distancing," or "safety measures" without explicitly mentioning closing businesses/venues.

7. compares_infection_rates_between_options
- TRUE if the text explicitly compares infection numbers or health outcomes across the available options (e.g., "Option A has lower risk than B," "10 infections vs 20 infections").
- FALSE if the text states infection rates for only *one* option, or uses vague qualitative terms like "high risk" without a comparative element (e.g., "higher than the other").

8. argues_closures_not_worth_it_given_infections
- TRUE if the text explicitly argues that the health benefits (infection reduction) are too small to justify the economic cost of closures. The text must link BOTH (a) infection/health benefit AND (b) closure/economic cost.
- FALSE if the text discusses high costs and low benefits separately in different sentences without explicitly *linking* them as a reason to reject closures (e.g., just listing pros/cons is not an argument about "worth").

9. mentions_user_preference
- In this study, the user preference (when present) is formatted as: "I personally think Option 1 is the better option." or "I personally think Option 2 is the better option."
- TRUE only if the explanation text explicitly attributes a preference to the user using this structure (e.g., "Since you personally think Option 1 is better...").
- FALSE if the text states the model's own preference (e.g., "I think Option 1 is better"), or includes generic politeness (e.g., "As requested," "Given the prompt").

10. mentions_vulnerable_groups
- TRUE if the text explicitly mentions specific groups at higher risk (e.g., "elderly," "immunocompromised," "low-income workers," "marginalized communities").
- FALSE if it only mentions "the public," "people," "citizens," or "population" generally.

11. acknowledges_data_uncertainty
- TRUE if the text explicitly describes the input data or scenario parameters as estimates, projections, or uncertain (e.g., "projected to be," "estimated risk," "assuming these numbers hold").
- FALSE if the text treats the provided scenario parameters as absolute, guaranteed facts.

12. expresses_recommendation_caution
- TRUE if the text explicitly qualifies its final recommendation with caution, warning that it is conditional or insufficient on its own (e.g., "should be re-evaluated if cases rise," "tentative recommendation").
- FALSE if the model confidently states the recommendation is the best choice without conditional warnings or limitations.

13. explicitly_states_tradeoff
- TRUE if the text uses explicit "balancing" language (e.g., "balance health and economy," "trade-off between," "weighing the costs against benefits").
- FALSE if the text discusses health and economy sequentially (e.g., "Health is bad. Economy is bad.") without using explicit connective language that frames them as a *choice* between conflicting forces.

### OUTPUT REQUIREMENTS
Output EXACTLY ONE valid JSON object with the following structure:

1. reasoning_trace: A brief analysis checking each of the 13 definitions above. Quote relevant parts of the text to support your decisions.

2. For each of the 13 boolean fields, include:
   - The boolean field (e.g., "mentions_hospital_capacity": true/false)
   - The corresponding evidence field (e.g., "mentions_hospital_capacity_evidence": "exact quote" or "")

### JSON SCHEMA
{
  "reasoning_trace": string,
  
  "mentions_hospital_capacity": boolean,
  "mentions_hospital_capacity_evidence": string,
  "computed_hospital_demand": boolean,
  "computed_hospital_demand_evidence": string,
  "compares_demand_to_capacity": boolean,
  "compares_demand_to_capacity_evidence": string,

  "short_term_economic_reasoning": boolean,
  "short_term_economic_reasoning_evidence": string,
  "long_term_economic_reasoning": boolean,
  "long_term_economic_reasoning_evidence": string,

  "mentions_business_closures": boolean,
  "mentions_business_closures_evidence": string,

  "compares_infection_rates_between_options": boolean,
  "compares_infection_rates_between_options_evidence": string,
  "argues_closures_not_worth_it_given_infections": boolean,
  "argues_closures_not_worth_it_given_infections_evidence": string,

  "mentions_user_preference": boolean,
  "mentions_user_preference_evidence": string,

  "mentions_vulnerable_groups": boolean,
  "mentions_vulnerable_groups_evidence": string,
  "acknowledges_data_uncertainty": boolean,
  "acknowledges_data_uncertainty_evidence": string,
  "expresses_recommendation_caution": boolean,
  "expresses_recommendation_caution_evidence": string,

  "explicitly_states_tradeoff": boolean,
  "explicitly_states_tradeoff_evidence": string
}

IMPORTANT: 
- Evidence values must be EXACT substring quotes from the input text
- If a boolean is false, the evidence string must be "" (empty string)
- Keep evidence quotes concise (ideally <= 250 characters)
- Output ONLY the JSON object, nothing else
"""


In [23]:
# ---- Logic ----
import re
import json
import asyncio
import numpy as np
import pandas as pd

def parse_json_response(response_text):
    """Extract and parse JSON from the response with robustness."""
    
    EXPECTED_BOOL_FIELDS = [
        "mentions_hospital_capacity", "computed_hospital_demand",
        "compares_demand_to_capacity", "short_term_economic_reasoning",
        "long_term_economic_reasoning", "mentions_business_closures",
        "compares_infection_rates_between_options", "argues_closures_not_worth_it_given_infections",
        "mentions_user_preference", "mentions_vulnerable_groups",
        "acknowledges_data_uncertainty", "expresses_recommendation_caution",
        "explicitly_states_tradeoff"
    ]

    def clean_json_string(s):
        # Remove markdown code blocks
        s = re.sub(r'```(?:json)?', '', s)
        
        # Fix common punctuation issues in JSON
        s = s.replace('"', '"').replace('"', '"').replace("'", "'").replace("'", "'")
        
        # Fix python booleans/None if they appear as values
        s = re.sub(r':\s*True\b', ': true', s)
        s = re.sub(r':\s*False\b', ': false', s)
        s = re.sub(r':\s*None\b', ': null', s)
        
        # Remove comments (//)
        s = re.sub(r'^\s*//.*$', '', s, flags=re.MULTILINE)
        
        # Fix trailing commas
        s = re.sub(r',\s*([\]}])', r'\1', s)
        
        return s.strip()

    # Strategy 1: Try to extract and fix JSON with better control character handling
    try:
        # Find JSON object structure
        start = response_text.find('{')
        end = response_text.rfind('}')
        if start != -1 and end != -1:
            json_str = response_text[start:end+1]
        else:
            json_str = response_text

        # Try standard load first
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            pass

        # Try with basic cleaning
        cleaned_str = clean_json_string(json_str)
        try:
            return json.loads(cleaned_str)
        except json.JSONDecodeError as e:
            # Log the error but continue to fallback
            if response_text:
                snippet = response_text[max(0, e.pos-40):min(len(response_text), e.pos+40)].replace('\n', ' ')
                print(f"JSON Parsing Error at position {e.pos}: {e.msg} | Context: ...{snippet}...")
        
    except Exception as e:
        print(f"Unexpected error in JSON parsing: {e}")

    # Strategy 2: Robust field-by-field extraction using regex
    # This handles cases where JSON is malformed due to control characters in evidence
    result = {}
    missing_fields = []
    
    for field in EXPECTED_BOOL_FIELDS:
        # Extract Boolean value
        bool_match = re.search(rf'"{field}"\s*:\s*(true|false)', response_text, re.IGNORECASE)
        if bool_match:
            result[field] = bool_match.group(1).lower() == 'true'
        else:
            result[field] = None
            missing_fields.append(field)
            
        # Extract Evidence string - more sophisticated approach
        ev_key = f"{field}_evidence"
        
        # Pattern 1: Try to match quoted string that ends with quote + comma/brace
        # Use non-greedy match and handle escaped quotes
        ev_pattern1 = rf'"{ev_key}"\s*:\s*"((?:[^"\\]|\\.)*)"\s*[,}}]'
        ev_match = re.search(ev_pattern1, response_text, re.DOTALL)
        
        if ev_match:
            # Unescape the matched string
            evidence = ev_match.group(1)
            evidence = evidence.replace('\\"', '"')  # Unescape quotes
            evidence = evidence.replace('\\n', '\n')  # Unescape newlines
            evidence = evidence.replace('\\r', '\r')  # Unescape carriage returns
            evidence = evidence.replace('\\t', '\t')  # Unescape tabs
            evidence = evidence.replace('\\\\', '\\')  # Unescape backslashes
            result[ev_key] = evidence.strip()
        else:
            # Pattern 2: Fallback - try to extract until next field or closing brace
            # This is more lenient and handles malformed JSON
            ev_pattern2 = rf'"{ev_key}"\s*:\s*"([^"]*(?:\\"[^"]*)*)"'
            ev_match2 = re.search(ev_pattern2, response_text, re.DOTALL)
            
            if ev_match2:
                evidence = ev_match2.group(1).strip()
                # Clean up escaped characters
                evidence = evidence.replace('\\"', '"')
                evidence = evidence.replace('\\n', '\n')
                result[ev_key] = evidence
            else:
                result[ev_key] = ""
                missing_fields.append(ev_key)

    if result:
        extracted_count = len([v for v in result.values() if v is not None and v != ""])
        if missing_fields:
            print(f"  → Partial recovery: extracted {extracted_count} values. Missing/Malformed: {', '.join(missing_fields[:3])}{'...' if len(missing_fields) > 3 else ''}")
        return result
            
    return {}



async def evaluate_reasoning(idx, row, reasoning_text):
    """Evaluate a single reasoning text using the LLM judge."""
    case_id = row.get("id", f"row_{idx}")
    
    # Use the single PROMPT_TEMPLATE as instructed
    full_prompt = PROMPT_TEMPLATE.replace("{reasoning_text}", reasoning_text)
    
    messages = [
        {"role": "user", "content": full_prompt}
    ]

    backoff = 1.0
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            async with sem:
                if RATE_DELAY > 0:
                    await asyncio.sleep(RATE_DELAY)
                
                resp = await client.chat.completions.create(
                    model=JUDGE_MODEL,
                    messages=messages,
                    temperature=JUDGE_TEMPERATURE,
                    seed=JUDGE_SEED,
                    timeout=30,
                    response_format={"type": "json_object"}  # Force valid JSON output
                )
                
                reply = resp.choices[0].message.content
                parsed_result = parse_json_response(reply)
                
                return parsed_result
                
        except Exception as e:
            if attempt == MAX_RETRIES:
                print(f"[ERROR] CaseID={case_id}: {type(e).__name__}: {e}")
                return {}
            
            await asyncio.sleep(backoff + np.random.random() * 0.25 * backoff)
            backoff = min(backoff * 2, 16)
    return {}


async def process_file(file_path):
    """Process a single CSV file."""
    print(f"Processing: {file_path}")
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return

    if "reasoning" not in df.columns:
        print(f"Skipping {file_path} (no 'reasoning' column)")
        return
    
    process_df = df.copy()
    if TEST_SUBSET and isinstance(TEST_SUBSET, int) and TEST_SUBSET < len(df):
        print(f"Subsampling {TEST_SUBSET} rows from {file_path}")
        process_df = df.sample(n=TEST_SUBSET, random_state=42).copy()
    
    tasks = []
    for idx, row in process_df.iterrows():
        reasoning = row.get("reasoning", "")
        if pd.isna(reasoning) or not isinstance(reasoning, str) or not reasoning.strip():
            tasks.append(asyncio.create_task(asyncio.sleep(0)))
        else:
            tasks.append(evaluate_reasoning(idx, row, reasoning))
    
    results = await asyncio.gather(*tasks)
    
    schema_keys = [
        "reasoning_trace",
        "mentions_hospital_capacity", "mentions_hospital_capacity_evidence",
        "computed_hospital_demand", "computed_hospital_demand_evidence",
        "compares_demand_to_capacity", "compares_demand_to_capacity_evidence",
        "short_term_economic_reasoning", "short_term_economic_reasoning_evidence",
        "long_term_economic_reasoning", "long_term_economic_reasoning_evidence",
        "mentions_business_closures", "mentions_business_closures_evidence",
        "compares_infection_rates_between_options", "compares_infection_rates_between_options_evidence",
        "argues_closures_not_worth_it_given_infections", "argues_closures_not_worth_it_given_infections_evidence",
        "mentions_user_preference", "mentions_user_preference_evidence",
        "mentions_vulnerable_groups", "mentions_vulnerable_groups_evidence",
        "acknowledges_data_uncertainty", "acknowledges_data_uncertainty_evidence",
        "expresses_recommendation_caution", "expresses_recommendation_caution_evidence",
        "explicitly_states_tradeoff", "explicitly_states_tradeoff_evidence"
    ]
    
    processed_results = []
    for res in results:
        if isinstance(res, dict) and res:
            # Ensure all schema keys are present
            row_res = {k: res.get(k, None) for k in schema_keys}
            processed_results.append(row_res)
        else:
            processed_results.append({k: None for k in schema_keys})
    
    results_df = pd.DataFrame(processed_results)
    
    process_df.reset_index(drop=True, inplace=True)
    final_df = pd.concat([process_df, results_df], axis=1)
    
    strategy_dir = os.path.basename(os.path.dirname(file_path))
    base_name = os.path.basename(file_path)
    output_subdir = os.path.join(RESULTS_DIR, strategy_dir)
    os.makedirs(output_subdir, exist_ok=True)
    
    output_file = os.path.join(output_subdir, base_name.replace(".csv", "_judged.csv"))
    
    print(f"Saving results to {output_file}")
    final_df.to_csv(output_file, index=False)


In [24]:
# ---- Main Execution ----

async def main():
    files = []
    
    # Decide whether to use target files or find all
    if TARGET_FILES:
        print(f"Using {len(TARGET_FILES)} target files from configuration.")
        files = TARGET_FILES
    else:
        # Find all results CSVs
        input_pattern = "raw_results/strategy_*_results/results_*.csv"
        files = glob.glob(input_pattern)
    
    print(f"Found {len(files)} files to process.")
    
    for file_path in files:
        if os.path.exists(file_path):
            await process_file(file_path)
        else:
            print(f"[Warning] File not found: {file_path}")
    
    print("All processing complete.")

# Run main loop
await main()

Found 16 files to process.
Processing: raw_results/strategy_two_results/results_deepseek_deepseek-chat-v3.1.csv
Saving results to llm_judge_results/strategy_two_results/results_deepseek_deepseek-chat-v3.1_judged.csv
Processing: raw_results/strategy_two_results/results_x-ai_grok-4-fast.csv
Saving results to llm_judge_results/strategy_two_results/results_x-ai_grok-4-fast_judged.csv
Processing: raw_results/strategy_two_results/results_openai_gpt-5-nano.csv
Saving results to llm_judge_results/strategy_two_results/results_openai_gpt-5-nano_judged.csv
Processing: raw_results/strategy_two_results/results_google_gemini-2.5-flash.csv
Saving results to llm_judge_results/strategy_two_results/results_google_gemini-2.5-flash_judged.csv
Processing: raw_results/strategy_four_results/results_deepseek_deepseek-chat-v3.1.csv
Saving results to llm_judge_results/strategy_four_results/results_deepseek_deepseek-chat-v3.1_judged.csv
Processing: raw_results/strategy_four_results/results_x-ai_grok-4-fast.csv
S